In [ ]:
"""
Classify a local image with Google Gemini 3.7 Flash via OpenRouter.
Returns strictly {"label": <one of LABELS>, "confidence": <0.0-1.0>}.

pip install openai
export OPENROUTER_API_KEY=sk-or-...
"""

import base64
import json
import mimetypes
import os
from pathlib import Path
import asyncio


from openai import OpenAI, AsyncOpenAI

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
MODEL = "google/gemini-3.7-flash"

LABELS = [
    "bar_chart",
    "line_chart",
    "pie_chart",
    "waterfall_chart",
    "scatter_plot",
    "area_chart",
    "table",
    "other",
]

SCHEMA = {
    "name": "chart_classification",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "label": {
                "type": "string",
                "enum": LABELS,
                "description": "The chart type shown in the image.",
            },
            "confidence": {
                "type": "number",
                "minimum": 0.0,
                "maximum": 1.0,
                "description": "Probability that the label is correct.",
            },
        },
        "required": ["label", "confidence"],
        "additionalProperties": False,
    },
}

SYSTEM_PROMPT = (
    "You classify chart images extracted from business documents. "
    "Return only the label and a calibrated confidence value. "
    "Use 'other' if the image is not a chart or the type is not in the list."
)


def image_to_data_url(path: str | Path) -> str:
    path = Path(path)
    mime = mimetypes.guess_type(path.name)[0] or "image/png"
    b64 = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{b64}"

### Sync

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

image_path = Path("../../../data/test_images/donut_01.png")

response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    max_tokens=200,
    response_format={"type": "json_schema", "json_schema": SCHEMA},
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Classify this chart."},
                {
                    "type": "image_url",
                    "image_url": {"url": image_to_data_url(image_path)},
                },
            ],
        },
    ],
    # Optional: cheaper/faster, since classification needs no long reasoning.
    extra_body={"reasoning": {"effort": "minimal"}},
)

In [ ]:
try:
    prediction = json.loads(response.choices[0].message.content)
    print(f"Prediction: {prediction['label']} (confidence: {prediction['confidence']:.2f})")
except json.JSONDecodeError:
    print("Error: Invalid JSON in LLM response")
    prediction = None

Prediction: pie_chart (confidence: 0.99)


In [28]:
cost = response.usage.cost
print(f"Cost: {cost*100:.3f}$ct for {response.usage.total_tokens} tokens")

Cost: 0.065$ct for 1362 tokens


### Async

In [ ]:
async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

image_paths = [
    Path("../../../data/test_images/donut_01.png"),
    Path("../../../data/test_images/NVIDIA_stock_chart.png"),
]

tasks = []
for image_path in image_paths:
    tasks.append(
        async_client.chat.completions.create(
            model=MODEL,
            temperature=0,
            max_tokens=200,
            response_format={"type": "json_schema", "json_schema": SCHEMA},
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Classify this chart."},
                        {
                            "type": "image_url",
                            "image_url": {"url": image_to_data_url(image_path)},
                        },
                    ],
                },
            ],
            extra_body={"reasoning": {"effort": "minimal"}},
        )
    )

In [ ]:
results = await asyncio.gather(*tasks)

for result in results:
    try:
        prediction = json.loads(result.choices[0].message.content)
        print(f"Prediction: {prediction['label']} (confidence: {prediction['confidence']:.2f})")
    except json.JSONDecodeError:
        print("Error: Invalid JSON in LLM response")
        prediction = None

    cost = result.usage.cost
    print(f"Cost: {cost*100:.3f}$ct for {result.usage.total_tokens} tokens")

Prediction: pie_chart (confidence: 0.99)
Cost: 0.068$ct for 1378 tokens
Prediction: line_chart (confidence: 0.90)
Cost: 0.073$ct for 1431 tokens


### With Sephamore

In [38]:
MAX_CONCURRENT = 3
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

image_paths = [
    Path("../../../data/test_images/donut_01.png"),
    Path("../../../data/test_images/NVIDIA_stock_chart.png"),
]

async def classify_image(image_path: Path):
    async with semaphore:
        response = await async_client.chat.completions.create(
            model=MODEL,
            temperature=0,
            max_tokens=200,
            response_format={"type": "json_schema", "json_schema": SCHEMA},
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Classify this chart."},
                        {
                            "type": "image_url",
                            "image_url": {"url": image_to_data_url(image_path)},
                        },
                    ],
                },
            ],
            extra_body={"reasoning": {"effort": "minimal"}},
        )
        return response

tasks = [classify_image(image_path) for image_path in image_paths]

In [39]:
results = await asyncio.gather(*tasks)